<a href="https://colab.research.google.com/github/marinasantiago1718/WikipediaGraph/blob/main/wikiVote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graduação em Ciência da Computação  
## Disciplina: Resolução de Problemas em Grafos  

**Docente:** Prof. Ricardo Carubbi  
**Instituição:** Universidade de Fortaleza  

---

# Caracterização Estrutural de Redes Reais  
## Verificação da Hipótese de Escala Livre

---

### Autores
- João Miguel Drumond
- Marina Maia  
- Tales Pimentel  
  

---

## 1. Introdução

Este projeto foi desenvolvido com o objetivo de aplicar os conhecimentos adquiridos na Unidade I da disciplina de Resolução de Problemas em Grafos, por meio da análise estrutural de uma rede real obtida da coleção SNAP.

A proposta central consiste em investigar se a rede selecionada apresenta propriedades associadas a grafos de escala livre, bem como discutir os limites dessa classificação dentro do domínio analisado.

---

## 2. Objetivos Específicos

1. Construir o grafo a partir de um dataset real do SNAP;
2. Calcular métricas estruturais fundamentais;
3. Estimar e analisar a distribuição de graus (escala log-log);
4. Comparar comportamentos entre domínios de rede;
5. Relacionar achados empíricos com conceitos teóricos de grafos complexos.

---

## 3. Dataset Selecionado

O dataset escolhido pertence à coleção disponibilizada pelo **Stanford Network Analysis Project (SNAP)**  (https://snap.stanford.edu/data/index.html).

Rede analisada:

**Wikipedia networks, articles, and metadata**, especificamente o subconjunto **"wiki-Vote"**, que representa conexões estruturais entre páginas da Wikipédia.

### Contexto da Rede

Na Wikipédia, usuários podem se candidatar ao cargo de administrador por meio de um processo denominado *Request for Adminship (RfA)*. Durante esse processo, membros da comunidade votam a favor ou contra a promoção do candidato.

Na rede modelada:

- Cada nó representa um usuário da Wikipédia;
- Uma aresta direcionada de $ i \rightarrow j $ indica que o usuário \( i \) votou no usuário \( j \).

Trata-se, portanto, de um grafo direcionado, representando interações sociais baseadas em votação.

---

### Estatísticas Estruturais do Dataset

- Número total de nós: 7.115
- Número total de arestas: 103.689
- Maior Componente Fracamente Conexa (WCC): 7.066 nós (99,3%)
- Maior Componente Fortemente Conexa (SCC): 1.300 nós (18,3%)
- Coeficiente médio de clusterização: 0.1409
- Número total de triângulos: 608.389
- Diâmetro da rede: 7
- Diâmetro efetivo (90%): 3.8

---

### Fonte

Leskovec, J.; Huttenlocher, D.; Kleinberg, J.  
*Signed Networks in Social Media*. CHI 2010.  
*Predicting Positive and Negative Links in Online Social Networks*. WWW 2010.

In [1]:
!pip install algs4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.9 MB/s eta 0:00:00


metodo para limpar data set



In [3]:
def clean_wiki_dataset(input_path, output_path):
    edges = []
    nodes = set()

    # 1. Ler o arquivo e coletar as arestas (pulando comentários)
    with open(input_path, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            v, w = map(int, line.split())
            edges.append((v, w))
            nodes.add(v)
            nodes.add(w)

    # 2. Mapear IDs originais para 0...V-1
    # Isso é necessário porque a classe Digraph usa uma lista/dict baseada no range(V)
    sorted_nodes = sorted(list(nodes))
    mapping = {node_id: i for i, node_id in enumerate(sorted_nodes)}

    num_v = len(sorted_nodes)
    num_e = len(edges)

    # 3. Salvar no formato esperado pela classe Digraph
    with open(output_path, 'w') as f:
        f.write(f"{num_v}\n")
        f.write(f"{num_e}\n")
        for v, w in edges:
            f.write(f"{mapping[v]} {mapping[w]}\n")

    print(f"Limpeza concluída!")
    print(f"Vértices: {num_v}, Arestas: {num_e}")
    print(f"Arquivo salvo em: {output_path}")

# Executar a limpeza
clean_wiki_dataset('Wiki-Vote.txt', 'Wiki-Vote-Clean.txt')

Limpeza concluída!
Vértices: 7115, Arestas: 103689
Arquivo salvo em: Wiki-Vote-Clean.txt


In [10]:
%%writefile digraph.py
from algs4.bag import Bag

class Digraph:
    def __init__(self, v=0, **kwargs):
        self.V = v
        self.E = 0
        self.adj = {}
        for v in range(self.V):
            self.adj[v] = Bag()

        if 'file' in kwargs:
            in_file = kwargs['file']
            self.V = int(in_file.readline())
            self.adj = {}
            for v in range(self.V):
                self.adj[v] = Bag()
            E_count = int(in_file.readline())
            for i in range(E_count):
                line = in_file.readline().split()
                if line:
                    v, w = line
                    self.add_edge(int(v), int(w))

    def __str__(self):
        return f"{self.V} vertices, {self.E} edges"

    def add_edge(self, v, w):
        v, w = int(v), int(w)
        if v not in self.adj: self.adj[v] = Bag()
        if w not in self.adj: self.adj[w] = Bag()
        self.adj[v].add(w)
        self.E += 1

Writing digraph.py


obter numero de vertices e arestas


In [11]:
from digraph import Digraph

with open('Wiki-Vote-Clean.txt', 'r') as f:
    g = Digraph(file=f)


print(f"Total de vértices: {g.V}")
print(f"Total de arestas: {g.E}")

Total de vértices: 7115
Total de arestas: 103689


densidade

In [12]:

v = g.V
e = g.E


densidade = e / (v * (v - 1))

print(f"A densidade do grafo é: {densidade:.5f}")

A densidade do grafo é: 0.00205


grau medio

In [13]:

grau_medio = g.E / g.V

print(f"O grau médio do grafo é: {grau_medio:.2f}")

O grau médio do grafo é: 14.57
